# Lecture 5 – Edge Detection, Morphological Operations, Corners & Lines

**Digital Image Processing · Spring 2026 · Week 5 (Mar 18)**
**Instructor:** Dr. Hugo Guillen Ramirez

> Follow along by running each cell in order.
> Cells marked **✏️ Exercise** contain skeleton code for you to complete.

---
## Setup & Imports
Run the two cells below to load all libraries and the test image.

In [ ]:
# 1. Standard library imports
import base64
import io
import math
import os
import time
import timeit
import warnings

# Global configurations / directory setup
warnings.filterwarnings('ignore')
os.makedirs('img', exist_ok=True)

# 2. Third-party core imports
import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
from PIL import Image
from scipy import ndimage
from scipy.ndimage import convolve, gaussian_filter, median_filter

# 3. scikit-image imports (with graceful fallback)
try:
    # Base modules
    from skimage import color, data, feature, filters, measure, morphology
    
    # Specific functions and aliases
    from skimage import data as skdata
    from skimage.data import shepp_logan_phantom
    from skimage.draw import disk as draw_disk, line as draw_line_sk, rectangle
    from skimage.feature import canny, corner_harris, corner_peaks
    from skimage.filters import gaussian, threshold_otsu
    from skimage.measure import LineModelND, ransac
    from skimage.morphology import diamond, disk, square
    from skimage.transform import hough_line, hough_line_peaks, resize
    from skimage.transform import resize as sk_resize
    from skimage.util import random_noise

    HAS_SKIMAGE = True
except ImportError:
    HAS_SKIMAGE = False
    print("scikit-image not found — synthetic images will be used.")

plt.rcParams.update({
    "figure.dpi": 100,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "font.family": "DejaVu Sans",
})

# ── Test image ───────────────────────────────────────────────────────────────
_B64_IMG = "..."

def load_test_image(size=(256, 256), name=None):
    """Return a grayscale float32 image in [0, 1]."""
    if HAS_SKIMAGE:
        if name is None or name.lower() == "punch":
            img_data = base64.b64decode(_B64_IMG)
            img_pil = Image.open(io.BytesIO(img_data)).convert("L")
            img = np.array(img_pil, dtype=np.float32) / 255.0
        elif name.lower() == "camera":
            img = skdata.camera().astype(np.float32) / 255.0
            img = sk_resize(img, size, anti_aliasing=True).astype(np.float32)
    else:
        y, x = np.ogrid[:size[0], :size[1]]
        cy, cx = size[0] / 2, size[1] / 2
        r = np.sqrt((x - cx)**2 + (y - cy)**2)
        img = (0.5 + 0.5 * np.sin(r * 0.4)).astype(np.float32)
    return img.astype(np.float32)

IMG = load_test_image()
print(f"Test image: shape={IMG.shape}, dtype={IMG.dtype}, range=[{IMG.min():.3f}, {IMG.max():.3f}]")
plt.imshow(IMG, cmap="gray", vmin=0, vmax=1)
plt.title("Test image"); plt.axis("off"); plt.show()

# ── Utility: side-by-side image + histogram comparison ───────────────────────
def show_histogram_comparison(img_orig, img_mod, mod_title="Modified"):
    """Display original vs modified image with their normalised histograms."""
    fig, axes = plt.subplots(2, 2, figsize=(10, 6))
    for ax, im, title in zip(axes[0], [img_orig, img_mod], ["Original", mod_title]):
        ax.imshow(im, cmap="gray", vmin=0, vmax=1)
        ax.set_title(title); ax.axis("off")
    for ax, im, title in zip(axes[1], [img_orig, img_mod], ["Original hist", f"{mod_title} hist"]):
        counts, bins = np.histogram(im, bins=256, range=(0., 1.))
        ax.bar(bins[:-1], counts / im.size, width=1/256, color="steelblue")
        ax.set_xlim(0, 1); ax.set_title(title)
    plt.tight_layout(); plt.show()

# ── Kernel utilities ─────────────────────────────────────────────────────────
def box_kernel(size):
    """Uniform box kernel of given size."""
    return np.ones((size, size), dtype=np.float32) / (size * size)

def gaussian_kernel_2d(size, sigma):
    """Normalised 2-D Gaussian kernel."""
    k = size // 2
    y, x = np.ogrid[-k:k+1, -k:k+1]
    h = np.exp(-(x**2 + y**2) / (2.0 * sigma**2))
    h /= h.sum()
    return h.astype(np.float32)

def gaussian_kernel_1d(size, sigma):
    """Normalised 1-D Gaussian kernel."""
    k = size // 2
    x = np.arange(-k, k + 1, dtype=np.float64)
    h = np.exp(-x**2 / (2.0 * sigma**2))
    return (h / h.sum()).astype(np.float32)

In [ ]:
DPI = 100
# Plotting defaults
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['image.cmap'] = 'gray'
plt.rcParams['axes.titlesize'] = 11

# ── Reusable save helper ──────────────────────────────────────────────
def save(fig, slide_num):
    """Save figure as img/slideN.png at 100 dpi and display."""
    path = f'img/slide{slide_num}.png'
    fig.savefig(path, dpi=DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    print(f'  ✓ saved {path}')

# ── Test image ────────────────────────────────────────────────────────
print(f"Test image | shape: {IMG.shape}  dtype: {IMG.dtype}  "
      f"range: [{IMG.min():.0f}, {IMG.max():.0f}]")
print("Setup complete.")

---
# Warm-Up Review

**Recall from last week:**
- Linear filtering (convolution / correlation)
- Box / Gaussian filters – smoothing to avoid aliasing
- Correlation as "matching" (dot product → NCC)
- Pyramid representations for scale variations

## Convolution Refresher
*(generates the figure shown on the slide)*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve

# Demonstrate convolution step-by-step on a small patch
patch = np.array([
    [10, 10, 10, 80, 80],
    [10, 10, 10, 80, 80],
    [10, 10, 10, 80, 80],
    [10, 10, 10, 80, 80],
    [10, 10, 10, 80, 80],
], dtype=float)

kernel_avg = np.ones((3, 3)) / 9.0          # averaging (smoothing)
kernel_dx  = np.array([[-1, 0, 1],
                       [-1, 0, 1],
                       [-1, 0, 1]], dtype=float)  # horizontal derivative

out_smooth = convolve(patch, kernel_avg)
out_deriv  = convolve(patch, kernel_dx)

# Define the grid layout visually: '.' means an empty space
layout = [
    ['a', 'b', 'd'],
    ['.', 'c', 'e']
]

# Create the figure using subplot_mosaic
fig, axes = plt.subplot_mosaic(layout, figsize=(9, 6))

# Map the mosaic keys to their respective data and titles
plot_data = {
    'a': (patch, 'Input patch'),
    'b': (kernel_avg, 'Box kernel'),
    'c': (out_smooth, ''),
    'd': (kernel_dx, 'Horizontal derivative'),
    'e': (out_deriv, '')
}

# Iterate through the dictionary to plot
for key, (arr, title) in plot_data.items():
    ax = axes[key]
    if key in ['b','d']:
        im = ax.imshow(arr)
    else:
        im = ax.imshow(arr, cmap='cividis')
    ax.set_title(title, fontsize=10)
    
    # Add text annotations
    for (j, i), val in np.ndenumerate(arr):
        ax.text(i, j, f'{val:.1f}', ha='center', va='center',
                fontsize=10, color='white' if val <= 10 and val != 0 and val!=1 else 'black')
    
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle('Convolution: smoothing vs. derivative kernel', fontweight='bold')
fig.tight_layout()

## The Test Image

In [ ]:
fig,ax = plt.subplots(figsize=(5, 5))
ax.imshow(IMG,cmap='gray')
d = IMG.shape[0]
ax.set_title('Johnny Silvercat (%dx%d)'%(d,d),fontsize=18)
ax.axis('off')
plt.show()

## What Is a Derivative of an Image?

The derivative measures *how fast* a function changes.
For a discrete row of pixels: $\frac{dI}{dx}\big|_i \approx I[i+1] - I[i]$

**Edge detection = finding where the intensity changes sharply.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle

# Extract the row and compute the derivative
row = IMG[250, :]  # a single row
deriv = np.diff(row)

# Create a figure with a GridSpec layout: 2 rows, 2 columns
# The width_ratios makes the 1D plots a bit wider than the image
fig = plt.figure(figsize=(10, 5))
gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1])

# Assign axes to the grid
ax_img = fig.add_subplot(gs[:, 0])           # Spans both rows in column 0
ax_prof = fig.add_subplot(gs[0, 1])          # Top right
ax_der = fig.add_subplot(gs[1, 1], sharex=ax_prof) # Bottom right (sharing x-axis)

# --- Left: Plot the Image ---
ax_img.imshow(IMG, cmap='gray')
ax_img.set_title('Original Image')
ax_img.axis('off')

# Add a red rectangle to highlight row 250
# xy=(x, y) of the top-left corner. We start at x=0, y=248 to give it a thickness of 4 pixels
img_width = IMG.shape[1]
rect = Rectangle((0, 248), width=img_width, height=4, 
                 linewidth=1.5, edgecolor='red', facecolor='none')
ax_img.add_patch(rect)

# --- Right Top: 1-D intensity profile ---
ax_prof.plot(row, 'b-', lw=1.2)
ax_prof.set_ylabel('Intensity I(x)')
ax_prof.set_title('1-D intensity profile (row 250)')
ax_prof.fill_between(range(len(row)), row, alpha=0.15)
# Hide x-axis tick labels for the top plot since they share the bottom axis
ax_prof.tick_params(labelbottom=False) 

# --- Right Bottom: Derivative ---
ax_der.plot(deriv, 'r-', lw=1.2)
ax_der.set_ylabel('dI/dx')
ax_der.set_xlabel('pixel x')
ax_der.set_title('Finite difference: peaks = edges')
ax_der.axhline(0, color='gray', ls='--', lw=0.7)

fig.tight_layout()
plt.show()

---
# Block 1 — Gradient-Based Edge Detection

Edges mark **boundaries** — object boundaries, surface creases, texture borders, shadows.
We find them by computing the **image gradient**:

$$\nabla I = \begin{pmatrix} \partial I/\partial x \\ \partial I/\partial y \end{pmatrix}$$

- **Magnitude** $|\nabla I| = \sqrt{I_x^2 + I_y^2}$ → edge strength
- **Direction** $\theta = \arctan(I_y / I_x)$ → edge orientation

## Sobel Gradients $G_x$ and $G_y$

In [ ]:
sobel_x = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=float)
sobel_y = sobel_x.T

Gx = convolve(IMG, sobel_x)
Gy = convolve(IMG, sobel_y)
mag = np.sqrt(Gx**2 + Gy**2)
direction = np.arctan2(Gy, Gx)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes = axes.flatten()
axes[0].imshow(Gx, cmap='RdBu_r'); axes[0].set_title('Gx (horizontal)',fontsize=16)
axes[1].imshow(Gy, cmap='RdBu_r'); axes[1].set_title('Gy (vertical)',fontsize=16)
for ax in axes: ax.axis('off')
fig.tight_layout()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes = axes.flatten()
axes[0].imshow(direction, cmap='hsv'); axes[0].set_title('Direction $\Theta$',fontsize=16)
axes[1].imshow(mag, cmap='hot'); axes[1].set_title('Gradient magnitude |∇I|',fontsize=16)
for ax in axes: ax.axis('off')
fig.tight_layout()

## Comparing Roberts, Prewitt, and Sobel

In [ ]:
roberts_x = np.array([[1, 0], [0, -1]], dtype=float)
roberts_y = np.array([[0, 1], [-1, 0]], dtype=float)
prewitt_x = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=float)
prewitt_y = prewitt_x.T

def edge_mag_dir(img, kx, ky):
    gx = convolve(img, kx)
    gy = convolve(img, ky)
    return np.sqrt(gx**2 + gy**2),np.arctan2(gy, gx)

mag_roberts,dir_roberts = edge_mag_dir(IMG, roberts_x, roberts_y)
mag_prewitt,dir_prewitt = edge_mag_dir(IMG, prewitt_x, prewitt_y)
mag_sobel,dir_sobel   = edge_mag_dir(IMG, sobel_x, sobel_y)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()
# axes[0].imshow(IMG, cmap='gray'); axes[0].set_title('Original')
axes[0].imshow(mag_roberts, cmap='hot'); axes[0].set_title('Roberts',fontsize=18)
axes[1].imshow(mag_prewitt, cmap='hot'); axes[1].set_title('Prewitt',fontsize=18)
axes[2].imshow(mag_sobel, cmap='hot');   axes[2].set_title('Sobel',fontsize=18)
axes[3].imshow(dir_roberts, cmap='hsv')
axes[4].imshow(dir_prewitt, cmap='hsv')
axes[5].imshow(dir_sobel, cmap='hsv')
for ax in axes: ax.axis('off')
# fig.suptitle('Comparison of gradient operators', fontweight='bold')
fig.tight_layout()

## Zoomed Comparison on a Crop

In [ ]:
IMG2 = IMG[200:400,:200]

roberts_x = np.array([[1, 0], [0, -1]], dtype=float)
roberts_y = np.array([[0, 1], [-1, 0]], dtype=float)
prewitt_x = np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=float)
prewitt_y = prewitt_x.T

def edge_mag_dir(img, kx, ky):
    gx = convolve(img, kx)
    gy = convolve(img, ky)
    return np.sqrt(gx**2 + gy**2),np.arctan2(gy, gx)

mag_roberts,dir_roberts = edge_mag_dir(IMG2, roberts_x, roberts_y)
mag_prewitt,dir_prewitt = edge_mag_dir(IMG2, prewitt_x, prewitt_y)
mag_sobel,dir_sobel   = edge_mag_dir(IMG2, sobel_x, sobel_y)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()
# axes[0].imshow(IMG, cmap='gray'); axes[0].set_title('Original')
axes[0].imshow(mag_roberts, cmap='hot'); axes[0].set_title('Roberts',fontsize=18)
axes[1].imshow(mag_prewitt, cmap='hot'); axes[1].set_title('Prewitt',fontsize=18)
axes[2].imshow(mag_sobel, cmap='hot');   axes[2].set_title('Sobel',fontsize=18)
axes[3].imshow(dir_roberts, cmap='hsv')
axes[4].imshow(dir_prewitt, cmap='hsv')
axes[5].imshow(dir_sobel, cmap='hsv')
for ax in axes: ax.axis('off')
# fig.suptitle('Comparison of gradient operators', fontweight='bold')
fig.tight_layout()

## Problem: Noise Ruins Simple Derivatives

Derivatives amplify high-frequency content **including noise**.
Solution: smooth first with a Gaussian, *then* differentiate.

In [ ]:
np.random.seed(42)
noisy = IMG + np.random.normal(0, 25, IMG.shape)

mag_noisy,dir_noisy   = edge_mag_dir(noisy, sobel_x, sobel_y)
smoothed    = gaussian_filter(noisy, sigma=2)
mag_smooth,dir_smooth  = edge_mag_dir(smoothed, sobel_x, sobel_y)

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
axes[0].imshow(noisy, cmap='gray');        axes[0].set_title('Noisy image')
axes[1].imshow(mag_noisy, cmap='hot');     axes[1].set_title('Sobel on noisy (garbage!)')
axes[2].imshow(smoothed, cmap='gray');     axes[2].set_title('Gaussian smoothed (σ=2)')
axes[3].imshow(mag_smooth, cmap='hot');    axes[3].set_title('Sobel on smoothed (clean)')
for ax in axes: ax.axis('off')
# fig.suptitle('Derivatives amplify noise → smooth first!', fontweight='bold')
fig.tight_layout()

## Derivative-of-Gaussian (DoG) Kernels

Because convolution is **associative** we can merge smoothing and
differentiation into a *single* kernel:

$$\frac{\partial}{\partial x}(G_\sigma * I) = \frac{\partial G_\sigma}{\partial x} * I$$

In [ ]:
def dog_kernel_x(sigma, half=None):
    if half is None:
        half = int(3 * sigma)
    ax = np.arange(-half, half + 1, dtype=float)
    X, Y = np.meshgrid(ax, ax)
    G = np.exp(-(X**2 + Y**2) / (2 * sigma**2))
    dGdx = -(X / sigma**2) * G
    return dGdx - dGdx.mean()
    # return dGdx

sigmas_dog = [.5, 1, 2]
fig, axes = plt.subplots(2, len(sigmas_dog), figsize=(12, 8))
for col, s in enumerate(sigmas_dog):
    k = dog_kernel_x(s)
    axes[0, col].imshow(k, cmap='RdBu_r')
    axes[0, col].set_title(f'DoG kernel σ={s:.1f}',fontsize=18)
    axes[0, col].axis('off')

    dog_out = convolve(IMG, k)
    axes[1, col].imshow(np.abs(dog_out), cmap='hot')
    axes[1, col].set_title(f'|DoG * I| σ={s}',fontsize=18)
    axes[1, col].axis('off')

# fig.suptitle('Derivative-of-Gaussian kernels at different scales', fontweight='bold')
fig.tight_layout()

## The Canny Edge Detector — Pipeline Stages

1. Gaussian smoothing → suppress noise
2. Gradient magnitude & direction → find edges
3. Non-Maximum Suppression → thin to 1 px
4. Hysteresis thresholding → keep strong, link weak

In [ ]:
# Show each stage of Canny manually
# img_norm = IMG / 255.0

# Stage 1: Gaussian smoothing
stage1 = gaussian_filter(IMG, sigma=1.5)

# Stage 2: Gradient magnitude + direction
gx_c = convolve(stage1, sobel_x / 255.0)
gy_c = convolve(stage1, sobel_y / 255.0)
mag_c = np.sqrt(gx_c**2 + gy_c**2)

# Stage 3: NMS (quantised, simplified)
def simple_nms(mag, angle):
    rows, cols = mag.shape
    nms = np.zeros_like(mag)
    angle_deg = np.rad2deg(angle) % 180
    for i in range(1, rows - 1):
        for j in range(1, cols - 1):
            a = angle_deg[i, j]
            if (0 <= a < 22.5) or (157.5 <= a < 180):
                n1, n2 = mag[i, j-1], mag[i, j+1]
            elif 22.5 <= a < 67.5:
                n1, n2 = mag[i-1, j+1], mag[i+1, j-1]
            elif 67.5 <= a < 112.5:
                n1, n2 = mag[i-1, j], mag[i+1, j]
            else:
                n1, n2 = mag[i-1, j-1], mag[i+1, j+1]
            nms[i, j] = mag[i, j] if mag[i, j] >= max(n1, n2) else 0
    return nms

dir_c = np.arctan2(gy_c, gx_c)
# NMS on a smaller crop for speed
crop = slice(100, 300), slice(100, 400)
nms_result = simple_nms(mag_c[crop], dir_c[crop])

# Stage 4: Full Canny
stage4 = canny(IMG, sigma=1.5, low_threshold=0.2, high_threshold=0.4)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
axes[0].imshow(stage1, cmap='gray');              axes[0].set_title('1. Gaussian smoothing',fontsize=16)
axes[1].imshow(mag_c, cmap='hot');                axes[1].set_title('2. Gradient magnitude',fontsize=16)
axes[2].imshow(nms_result, cmap='bwr');          axes[2].set_title('3. After NMS (crop)',fontsize=16)
axes[3].imshow(stage4, cmap='bwr');              axes[3].set_title('4. Final Canny output',fontsize=16)
for ax in axes: ax.axis('off')
# fig.suptitle('Canny pipeline stages', fontweight='bold')
fig.tight_layout()

## Non-Maximum Suppression (NMS) — Before & After

In [ ]:
# Show before and after NMS on a zoomed region
crop2 = slice(200, 300), slice(180, 300)
mag_crop = mag_c[crop2]
nms_crop = simple_nms(mag_c[crop2], dir_c[crop2])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(IMG[crop2], cmap='gray');  axes[0].set_title('Original (crop)',fontsize=16)
axes[1].imshow(mag_crop, cmap='hot');     axes[1].set_title('Gradient magnitude (thick ridges)',fontsize=16)
axes[2].imshow(nms_crop, cmap='bwr');     axes[2].set_title('After NMS (1-pixel wide)',fontsize=16)
for ax in axes: ax.axis('off')
# fig.suptitle('NMS thins gradient ridges to single-pixel edges', fontweight='bold')
fig.tight_layout()

## Hysteresis Thresholding

Two thresholds $k_{\text{lo}} < k_{\text{hi}}$:
- $\geq k_{\text{hi}}$: **strong** (always kept)
- $\in [k_{\text{lo}}, k_{\text{hi}})$: **weak** (kept only if connected to strong)
- $< k_{\text{lo}}$: suppressed

In [ ]:
# Demonstrate strong / weak / suppressed
mag_full,dir_full = edge_mag_dir(gaussian_filter(IMG, 1.5), sobel_x, sobel_y)
nms = simple_nms(mag_full, dir_full)
lo, hi = 0.2, 0.4

strong = nms >= hi
weak   = (nms >= lo) & (nms < hi)
suppressed = nms < lo

# Color-code: green=strong, yellow=weak, black=suppressed
vis = np.zeros((*IMG.shape, 3))
vis[strong] = [0, 0.8, 0]       # green
vis[weak]   = [0.9, 0.7, 0]     # yellow
# suppressed stays black

fig, axes = plt.subplots(1, 3, figsize=(10, 4.5))
axes[0].imshow(nms, cmap='bwr'); axes[0].set_title('Non-maximum suppression',fontsize=16)
axes[1].imshow(vis);                  axes[1].set_title('Green=strong; Yellow=weak;\n Black=suppressed',fontsize=16)
axes[2].imshow(canny(IMG, sigma=1.5, low_threshold=lo, high_threshold=hi),
               cmap='bwr');          axes[2].set_title('Final Canny (after linking)',fontsize=16)
for ax in axes: ax.axis('off')
fig.suptitle(f'Hysteresis: $k_{{lo}}$={lo}, $k_{{hi}}$={hi}', fontweight='bold',fontsize=18)
fig.tight_layout()

## Laplacian-of-Gaussian (LoG)

An alternative: instead of peaks in the 1st derivative, look for
**zero crossings** in the 2nd derivative.

$$\nabla^2 I = \frac{\partial^2 I}{\partial x^2} + \frac{\partial^2 I}{\partial y^2}$$

In [ ]:
def log_kernel(sigma, half=None):
    if half is None:
        half = int(4 * sigma)
    ax = np.arange(-half, half + 1, dtype=float)
    X, Y = np.meshgrid(ax, ax)
    r2 = X**2 + Y**2
    s2 = sigma**2
    k = -(1 / (np.pi * s2**2)) * (1 - r2 / (2 * s2)) * np.exp(-r2 / (2 * s2))
    return k - k.mean()

# Show LoG kernel and zero crossings
log_k = log_kernel(2,half=12)
log_out = convolve(IMG, log_k)
zero_cross = np.zeros_like(log_out, dtype=bool)
for i in range(1, log_out.shape[0]-1):
    for j in range(1, log_out.shape[1]-1):
        patch = log_out[i-1:i+2, j-1:j+2]
        if patch.min() < 0 < patch.max():
            zero_cross[i, j] = True

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
axes = axes.flatten()
axes[0].imshow(log_k, cmap='RdBu_r'); axes[0].set_title('LoG kernel (σ=2,k=25)\n"Mexican hat"',fontsize=16)
axes[1].imshow(log_out, cmap='hot'); axes[1].set_title('LoG * I',fontsize=16)
axes[2].imshow(zero_cross, cmap='bwr'); axes[2].set_title('Zero crossings',fontsize=16)
axes[3].imshow(canny(IMG, sigma=2,low_threshold=.08, high_threshold=.16), cmap='bwr'); axes[3].set_title('Canny (σ=2,lo=0.08,hi=0.16)',fontsize=16)
for ax in axes: ax.axis('off')
# fig.suptitle('LoG edge detection via zero crossings', fontweight='bold')
fig.tight_layout()

## ✏️ Exercise 1 — Canny Pipeline Exploration (~10 min)

1. Apply Canny to an image of your choice with default parameters.
2. Sweep $\sigma \in \{0.5, 1, 2, 3\}$ and $k_{\text{hi}} \in \{0.05, 0.10, 0.20\}$;
   display the resulting edge maps in a $4 \times 3$ grid.

In [ ]:
# ✏️ Exercise 1 — fill in the code below

sigmas_sweep = [0.5, 1, 2, 3]
hi_sweep = [0.05, 0.10, 0.20]

fig, axes = plt.subplots(len(sigmas_sweep), len(hi_sweep),
                          figsize=(12, 14))

for r, s in enumerate(sigmas_sweep):
    for c, h in enumerate(hi_sweep):
        lo = h / 2
        # TODO: compute Canny edge map with sigma=s, low_threshold=lo, high_threshold=h
        # e = ...
        # axes[r, c].imshow(e, cmap='gray')
        axes[r, c].set_title(f'σ={s}, hi={h}', fontsize=9)
        axes[r, c].axis('off')

fig.suptitle('Exercise 1: Canny parameter sweep', fontweight='bold')
fig.tight_layout()
plt.show()

---
# Block 2 — Morphological Operations

After edge detection (or thresholding) you typically get a **messy binary mask**:
small noise blobs, tiny holes, touching objects.

**Morphological operations** clean up binary images by probing them with a
small shape called a **structuring element** (SE).

| Operation | Effect |
|-----------|--------|
| **Dilation** ($\oplus$) | Grow foreground |
| **Erosion** ($\ominus$) | Shrink foreground |
| **Opening** | Remove small bright noise (erode → dilate) |
| **Closing** | Fill small dark holes (dilate → erode) |

## Closing Example
*(opening removes noise outside, closing fills holes inside)*

In [ ]:
# Full exercise 3 solution
np.random.seed(42)
mask = np.zeros((200, 300), dtype=bool)
rr, cc = draw_disk((80, 100), 40, shape=mask.shape)
mask[rr, cc] = True
rr, cc = draw_disk((120, 220), 30, shape=mask.shape)
mask[rr, cc] = True

noisy_mask = random_noise(mask.astype(float), mode='s&p', amount=0.05) > 0.5
se_ex3 = diamond(2)
step1 = morphology.dilation(noisy_mask, se_ex3).astype(float)
step2 = morphology.erosion(step1, se_ex3).astype(float)


fig, axes = plt.subplots(3, 1, figsize=(4, 8))
axes[0].imshow(noisy_mask, cmap='gray'); axes[0].set_title('Noisy image',fontsize=20)
axes[1].imshow(step1, cmap='gray');      axes[1].set_title('dilation',fontsize=20)
axes[2].imshow(step2, cmap='gray');      axes[2].set_title('dilation+erosion',fontsize=20)
for ax in axes: ax.axis('off')
fig.tight_layout()

## Opening Example

In [ ]:
# Full exercise 3 solution
np.random.seed(42)
mask = np.zeros((200, 300), dtype=bool)
rr, cc = draw_disk((80, 100), 40, shape=mask.shape)
mask[rr, cc] = True
rr, cc = draw_disk((120, 220), 30, shape=mask.shape)
mask[rr, cc] = True

noisy_mask = random_noise(mask.astype(float), mode='s&p', amount=0.05) > 0.5
se_ex3 = diamond(2)
step1 = morphology.erosion(noisy_mask, se_ex3).astype(float)
step2 = morphology.dilation(step1, se_ex3).astype(float)


fig, axes = plt.subplots(3, 1, figsize=(4, 8))
axes[0].imshow(noisy_mask, cmap='gray'); axes[0].set_title('Noisy image',fontsize=20)
axes[1].imshow(step1, cmap='gray');      axes[1].set_title('erosion',fontsize=20)
axes[2].imshow(step2, cmap='gray');      axes[2].set_title('erosion+dilation',fontsize=20)
for ax in axes: ax.axis('off')
fig.tight_layout()

## Combining Opening + Closing

In [ ]:
se_ex3 = diamond(2)
stepA = morphology.opening(noisy_mask, se_ex3).astype(float)
stepB = morphology.closing(noisy_mask, se_ex3).astype(float)
stepA1 = morphology.closing(stepA, se_ex3).astype(float)
stepB1 = morphology.opening(stepB, se_ex3).astype(float)

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
axes = axes.flatten()
axes[0].imshow(stepA, cmap='gray'); axes[0].set_title('opening',fontsize=20)
axes[1].imshow(stepA1, cmap='gray');      axes[1].set_title('opening+\nclosing',fontsize=20)
axes[2].imshow(stepB, cmap='gray');      axes[2].set_title('closing',fontsize=20)
axes[3].imshow(stepB1, cmap='gray');      axes[3].set_title('closing+\nopening',fontsize=20)
for ax in axes: ax.axis('off')
fig.tight_layout()

## Morphological Gradient (Edge Detector)

$$\text{grad}(I) = (I \oplus B) - (I \ominus B)$$

Dilated minus eroded = thick edges at boundaries.

In [ ]:
se_mg = disk(3)
morph_grad = morphology.dilation(IMG, se_mg).astype(float) - \
             morphology.erosion(IMG, se_mg).astype(float)

sobel_mag,sobel_dir = edge_mag_dir(IMG, sobel_x, sobel_y)

fig, axes = plt.subplots(1, 3, figsize=(10, 4))
axes[0].imshow(IMG, cmap='gray');        axes[0].set_title('Original',fontsize=20)
axes[1].imshow(morph_grad, cmap='hot');  axes[1].set_title('dilation − erosion',fontsize=20)
axes[2].imshow(sobel_mag, cmap='hot');   axes[2].set_title('Sobel mag on original',fontsize=20)
for ax in axes: ax.axis('off')
fig.tight_layout()

## ✏️ Exercise 2 — Morphological Clean-Up (~10 min)

Apply morphological operations to clean the noisy mask below.

**Hint:** try opening followed by closing (or vice-versa) with a diamond(2) or disk(3) SE.

In [ ]:
# ✏️ Exercise 2 — fill in the code below

from skimage.morphology import diamond, disk
from skimage.util import random_noise

###############################################################################
###############################################################################
# IMAGE CREATION
# Create a noisy binary mask
np.random.seed(42)
mask = np.zeros((400, 400), dtype=bool)

# scattered cells / blobs
for _ in range(15):
    cy, cx = np.random.randint(40, 360, size=2)
    r = np.random.randint(12, 35)
    rr, cc = draw_disk((cy, cx), r, shape=mask.shape)
    mask[rr, cc] = True

# a long curved "vessel"
t = np.linspace(0, 2 * np.pi, 800)
yy = (200 + 120 * np.sin(t)).astype(int)
xx = (np.linspace(30, 370, len(t))).astype(int)
for dy in range(-3, 4):
    mask[np.clip(yy + dy, 0, 399), xx] = True

# rectangular "tissue boundary"
mask[280:320, 50:350] = True
mask[280:395, 160:200] = True

# punch holes inside objects
for _ in range(30):
    cy, cx = np.random.randint(0, 400, size=2)
    if mask[cy, cx]:
        rr, cc = draw_disk((cy, cx), np.random.randint(2, 6), shape=mask.shape)
        mask[rr, cc] = False

# add salt-and-pepper + small blob noise
noisy_mask = random_noise(mask.astype(float), mode='s&p', amount=0.06) > 0.5
for _ in range(40):
    cy, cx = np.random.randint(0, 400, size=2)
    rr, cc = draw_disk((cy, cx), np.random.randint(1, 4), shape=noisy_mask.shape)
    noisy_mask[rr, cc] = not mask[cy, cx]

###############################################################################
###############################################################################
# YOUR WORK STARTS HERE

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(noisy_mask, cmap='gray'); axes[0].set_title('Noisy mask')

# TODO: apply morphological operations to clean the mask
# se = ...
# cleaned = ...

# TODO: plot result image (just uncomment)
# axes[1].imshow(cleaned, cmap='gray'); axes[1].set_title('After cleaning')

axes[2].imshow(mask, cmap='gray'); axes[2].set_title('Ground truth')
for ax in axes: ax.axis('off')
fig.tight_layout()
plt.show()

---
# Block 3 — Corners & Lines

## The Aperture Problem

- **Flat region** → slide any direction → no change → *boring*
- **Edge** → slide along → no change; slide across → change → *ambiguous*
- **Corner** → slide any direction → change → *distinctive!*

Corners are ideal anchor points for matching, tracking, and stitching.

## Aperture Problem — Visual

In [ ]:
"""
Harris Corner Detector — Gradient Distribution Plots
=====================================================
Generates scatter plots of (Ix, Iy) gradient vectors for four
synthetic patch types: Flat, Linear Edge, Corner, and Vertical Edge.
Each plot includes the principal-component ellipse fitted to the
gradient cloud, matching the style from Hebert / Collins slides.

Usage: run all cells in a Jupyter notebook, or run as a script.
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
# CHANGED: Import correlate instead of convolve
from scipy.ndimage import correlate, gaussian_filter

# ---------------------------------------------------------------------------
# 1.  Helper: compute Ix, Iy with Sobel
# ---------------------------------------------------------------------------
def sobel_gradients(patch):
    """Return (Ix, Iy) arrays for a greyscale patch."""
    Gx = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]], dtype=float) / 8.0   # normalise
    Gy = Gx.T
    
    # CHANGED: Use correlate to prevent the kernel from flipping.
    # This ensures transitions from dark (0) to bright (1) yield positive gradients.
    Ix = correlate(patch.astype(float), Gx)
    Iy = correlate(patch.astype(float), Gy)
    return Ix, Iy

# ---------------------------------------------------------------------------
# 2.  Helper: PCA ellipse from a 2-column data matrix
# ---------------------------------------------------------------------------
def pca_ellipse(Ix_flat, Iy_flat, n_std=2.0):
    """Return (centre, width, height, angle_deg) for matplotlib Ellipse."""
    D = np.column_stack([Ix_flat, Iy_flat])
    
    # CHANGED: Harris uses the uncentered Structure Tensor (Second Moment Matrix).
    # We use (D.T @ D) / N instead of np.cov, which centers the data.
    M = (D.T @ D) / len(D) 
    eigvals, eigvecs = np.linalg.eigh(M)
    
    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    
    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
    width  = 2 * n_std * np.sqrt(max(eigvals[0], 1e-10))
    height = 2 * n_std * np.sqrt(max(eigvals[1], 1e-10))
    
    # CHANGED: Force center to origin, consistent with Structure Tensor math
    return (0.0, 0.0), width, height, angle

# ---------------------------------------------------------------------------
# 3.  Build four synthetic patches  (64 x 64, values in [0, 1])
# ---------------------------------------------------------------------------
np.random.seed(42)
sz = 64
noise_std = 0.015       # subtle noise — keeps flat region tight

def add_noise(patch):
    return patch + noise_std * np.random.randn(*patch.shape)

# --- Flat: uniform grey + light noise
flat_patch = add_noise(0.5 * np.ones((sz, sz)))

# --- Horizontal edge (dark top, bright bottom) — "Linear Edge"
hedge_base = np.zeros((sz, sz))
hedge_base[sz // 2:, :] = 1.0
hedge_base = gaussian_filter(hedge_base, sigma=1.2)
hedge_patch = add_noise(hedge_base)

# --- Corner (dark top-left quadrant, bright elsewhere)
corner_base = np.ones((sz, sz))
corner_base[: sz // 2, : sz // 2] = 0.0
corner_base = gaussian_filter(corner_base, sigma=1.2)
corner_patch = add_noise(corner_base)

# --- Vertical edge (dark left, bright right)
vedge_base = np.zeros((sz, sz))
vedge_base[:, sz // 2:] = 1.0
vedge_base = gaussian_filter(vedge_base, sigma=1.2)
vedge_patch = add_noise(vedge_base)

patches = {
    "Flat":           flat_patch,
    "H. Edge":    hedge_patch,
    "Corner":         corner_patch,
    "V. Edge":  vedge_patch,
}

# ---------------------------------------------------------------------------
# 4.  Eigenvalue annotations for each case
# ---------------------------------------------------------------------------
eigen_labels = {
    "Flat":          r"$\lambda_1 \approx \lambda_2 \approx$ small",
    "H. Edge":   r"$\lambda_1$ large; $\lambda_2 \approx$ small",
    "Corner":        r"$\lambda_1 \approx \lambda_2 \approx$ large",
    "V. Edge": r"$\lambda_1$ large; $\lambda_2 \approx$ small",
}

# ---------------------------------------------------------------------------
# 5.  Plot
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
axes = axes.ravel()

for idx, (name, patch) in enumerate(patches.items()):
    ax = axes[idx]

    # Gradients
    Ix, Iy = sobel_gradients(patch)

    # Crop 2-pixel border to avoid boundary artefacts
    Ix_inner = Ix[2:-2, 2:-2].ravel()
    Iy_inner = Iy[2:-2, 2:-2].ravel()

    # Scatter
    ax.scatter(Ix_inner, Iy_inner, s=5, alpha=0.55, color="royalblue",
               edgecolors="none", zorder=2)

    # PCA ellipse
    mu, w, h, angle = pca_ellipse(Ix_inner, Iy_inner, n_std=2.5)
    ell = Ellipse(xy=mu, width=w, height=h, angle=angle,
                  edgecolor="red", facecolor="none", linewidth=2.0)
    ax.add_patch(ell)

    # Axes through origin — auto-scale per subplot
    lim = max(0.15, 1.3 * max(abs(Ix_inner).max(), abs(Iy_inner).max()))
    ax.axhline(0, color="k", linewidth=1.0)
    ax.axvline(0, color="k", linewidth=1.0)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.set_xlabel(r"$I_x$", fontsize=14)
    ax.set_ylabel(r"$I_y$", fontsize=14)
    ax.tick_params(labelsize=9)

    # Title + eigenvalue annotation
    ax.set_title(name, fontsize=16, fontweight="bold", pad=10)
    ax.text(0.03, 0.03, eigen_labels[name], transform=ax.transAxes,
            fontsize=12, verticalalignment="bottom",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray",
                      alpha=0.9))

    # Inset: the original patch
    inset = ax.inset_axes([0.6, 0.6, 0.38, 0.38])
    inset.imshow(patch, cmap="gray", vmin=0, vmax=1)
    inset.set_xticks([])
    inset.set_yticks([])
    for spine in inset.spines.values():
        spine.set_edgecolor("darkorange")
        spine.set_linewidth(2.5)

plt.tight_layout(pad=2.0)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
axes = axes.flatten()
for idx, (name, patch) in enumerate(patches.items()):
    ax = axes[idx]
    ax.set_title(name, fontsize=16, fontweight="bold")
    ax.imshow(patch)
    D = patch.shape[0]    
    rect = plt.Rectangle((0,0), D-1,D-1, fill=False, color="orange", linewidth=3)    
    ax.add_patch(rect)
    ax.axis('off')

# plt.tight_layout()
plt.show()

## Harris Corner Detector

$$R = \det(M) - k\,(\operatorname{tr} M)^2$$

- $R \gg 0$: **corner**
- $R \ll 0$: edge
- $|R| \approx 0$: flat region

In [ ]:
checker = data.checkerboard().astype(float)
# 1. Compute the Harris response map
R = corner_harris(checker, k=0.05, sigma=1.0)
# 2. Find corner locations (peaks in R)
corners = corner_peaks(R, min_distance=10,threshold_rel=0.02)


fig, axes = plt.subplots(1, 3, figsize=(12, 4.5))
axes[0].imshow(checker, cmap='gray'); axes[0].set_title('Original',fontsize=20)
axes[1].imshow(R, cmap='RdBu_r'); axes[1].set_title('Harris R',fontsize=20)
axes[2].imshow(checker, cmap='gray')
axes[2].plot(corners[:, 1], corners[:, 0], 'r+', markersize=12, mew=1.5)
axes[2].set_title(f'Detected corners ({len(corners)})',fontsize=20)
for ax in axes: ax.axis('off')
fig.tight_layout()
plt.show()

## Hough Transform for Lines

Each edge pixel casts a "vote" for every candidate line through it.
Peaks in the accumulator = strong lines.

$$x\cos\theta + y\sin\theta = \rho$$

In [ ]:
# Create a clean synthetic line image
np.random.seed(42)
line_img = np.zeros((200, 300), dtype=np.uint8)
rr, cc = draw_line_sk(20, 30, 180, 270)
rr = np.clip(rr, 0, 199); cc = np.clip(cc, 0, 299)
line_img[rr, cc] = 255
rr2, cc2 = draw_line_sk(10, 250, 190, 50)
rr2 = np.clip(rr2, 0, 199); cc2 = np.clip(cc2, 0, 299)
line_img[rr2, cc2] = 255
rr3, cc3 = draw_line_sk(140, 10, 140, 290)
rr3 = np.clip(rr3, 0, 199); cc3 = np.clip(cc3, 0, 299)
line_img[rr3, cc3] = 255

# Add noise
noisy_lines = line_img.astype(float) + np.random.normal(0, 20, line_img.shape)
edge_lines = canny(noisy_lines / 255.0, sigma=1)

tested_angles = np.linspace(-np.pi/2, np.pi/2, 360, endpoint=False)
h_space, theta_arr, dist_arr = hough_line(edge_lines, theta=tested_angles)

# 1. Stricter peak extraction to ignore noise
# We increase the minimum thresholds and explicitly limit to our top 3 lines
hspace_peaks, angles, dists = hough_line_peaks(
    h_space, 
    theta_arr, 
    dist_arr, 
    min_distance=20, 
    min_angle=15,
    threshold=0.4 * np.max(h_space), # Higher threshold filters out weaker noise lines
    num_peaks=3                     # Forces it to only keep the top 3 strongest votes
)

# 2. Create the enhanced visualization
fig, axes = plt.subplots(2,2,figsize=(10, 8))
axes = axes.flatten()

axes[0].imshow(noisy_lines, cmap='gray'); axes[0].set_title('Noisy lines',fontsize=18)
######################################################################
axes[1].imshow(edge_lines, cmap='gray'); axes[1].set_title('Canny',fontsize=18)
######################################################################

###########################################################
axes[2].set_facecolor('black')
axes[2].imshow(np.log1p(h_space), aspect='auto', cmap='viridis', alpha=1,
          extent=[np.rad2deg(theta_arr[0]), np.rad2deg(theta_arr[-1]),
                  dist_arr[-1], dist_arr[0]])
# Overlay the detected peaks as sharp, distinct points
angles_deg = np.rad2deg(angles)
axes[2].plot(angles_deg, dists, 'co', markersize=8, markeredgecolor='white', 
        label=f'{len(angles)} Lines Detected')

axes[2].set_xlabel('θ (degrees)')
axes[2].set_ylabel('ρ (pixels)')
axes[2].set_title('Hough accumulator space',fontsize=18)
axes[2].legend()
################################################
axes[3].imshow(noisy_lines, cmap='gray'); axes[3].set_title('Found lines',fontsize=18)
for angle, dist in zip(angles, dists):
    (x0, y0) = dist * np.array([np.cos(angle), np.sin(angle)])
    axes[3].axline((x0, y0), slope=np.tan(angle + np.pi/2),
                   color='red', linewidth=1.5, alpha=0.8)

axes[3].set_xlim(0, noisy_lines.shape[1])
axes[3].set_ylim(noisy_lines.shape[0], 0)
for i in [0,1,3]: axes[i].axis('off')

fig.tight_layout()
plt.show()

## ✏️ Exercise 3 — Corners, Hough & RANSAC (~5 min)

1. Apply the Harris detector to Johnny Silvercat; overlay detected corners.
2. Detect 15 lines with the Hough transform; overlay them.

In [ ]:
# ✏️ Exercise 3 — fill in the code below

from skimage.feature import corner_harris, corner_peaks
from skimage.transform import hough_line, hough_line_peaks

R = corner_harris(IMG, k=0.05, sigma=1.0)
# TODO: find corner locations using corner_peaks
# corners = corner_peaks(R, ...)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(IMG, cmap='gray')
# TODO: overlay corners (just uncomment)
# ax.plot(corners[:, 1], corners[:, 0], 'r+', markersize=8)
ax.set_title('Harris corners and \nHough lines on Johnny Silvercat',fontsize=18)


# Hough lines
edge_lines = canny(IMG, sigma=1)
tested_angles = np.linspace(-np.pi/2, np.pi/2, 360, endpoint=False)
h_space, theta_arr, dist_arr = hough_line(edge_lines, theta=tested_angles)

# TODO: find 15 line peaks with hough_line_peaks
# hspace_peaks, angles, dists = hough_line_peaks( ...

# TODO: overlay lines (just uncomment)
# for angle, dist in zip(angles, dists):
#     (x0, y0) = dist * np.array([np.cos(angle), np.sin(angle)])
#     ax.axline((x0, y0), slope=np.tan(angle + np.pi/2),
                   # color='blue', linewidth=1, alpha=0.5)

ax.axis('off')
plt.show()

---
# Summary

| 1. Edge Detection | 2. Morphology | 3. Corners | 4. Lines & Models |
|---|---|---|---|
| Gradients highlight intensity changes | Clean up binary masks with SEs | Harris: gradient strong in *multiple* directions | Vote in parameter space (Hough) or sample (RANSAC) |
| Canny = smooth → gradient → NMS → hysteresis | Dilation / Erosion / Opening / Closing | $R = \det(M) - k(\operatorname{tr}M)^2$ | $x\cos\theta + y\sin\theta = \rho$ |

## Learning Goals — Self-Check

Can you now…

1. ✅ **Explain** what an image gradient is and how it relates to edges.
2. ✅ **Write** the Sobel kernels from memory and apply them to a patch.
3. ✅ **Name** the four steps of the Canny pipeline and explain what each one does.
4. ✅ **Distinguish** dilation, erosion, opening, and closing — and know when to use each.
5. ✅ **Clean up** a noisy binary mask using morphological operations in code.
6. ✅ **Explain** why corners are more distinctive than edges, and what the Harris response $R$ tells you.
7. ✅ **Describe** how the Hough transform uses voting to detect lines.